In [ ]:
'''#1. Disinstalliamo TensorFlow (il colpevole principale) e Protobuf
!pip uninstall -y tensorflow tensorflow-probability protobuf

# 2. Reinstalliamo Protobuf in una versione "sicura" (3.20.x)
!pip install "protobuf==3.20.3"

# 3. Installiamo vLLM e le altre librerie
!pip install vllm numpy transformers accelerate

print("--- PULIZIA COMPLETATA ---")
print("ORA DEVI RIAVVIARE LA SESSIONE (Vedi passo 2)")'''

'#1. Disinstalliamo TensorFlow (il colpevole principale) e Protobuf\n!pip uninstall -y tensorflow tensorflow-probability protobuf\n\n# 2. Reinstalliamo Protobuf in una versione "sicura" (3.20.x)\n!pip install "protobuf==3.20.3"\n\n# 3. Installiamo vLLM e le altre librerie\n!pip install vllm numpy transformers accelerate\n\nprint("--- PULIZIA COMPLETATA ---")\nprint("ORA DEVI RIAVVIARE LA SESSIONE (Vedi passo 2)")'

In [ ]:
# ==========================================
# SEZIONE 2: IMPORT E SETUP
# ==========================================
import json
import re
import numpy as np
from google.colab import drive

# Monta Google Drive
drive.mount('/content/drive')

# Percorsi File
INPUT_PATH = '/content/drive/MyDrive/results_enhanced_summary_0.json'
OUTPUT_PATH = '/content/drive/MyDrive/results_evaluated_llama_vllm.json'

# Definizione dei Prompt G-Eval (Ottimizzati per Qwen)
GEVAL_PROMPTS = {
    "faithfulness": """You are an expert summary evaluator.
Task: Evaluate the Faithfulness (1-5) of the Summary based on the Source Text.
GOAL: Determine if the summary contains *only* information supported by the source.

Input Data:
<source_text>
{source}
</source_text>

<summary_text>
{generated}
</summary_text>

Evaluation Steps:
1. Read the summary sentence by sentence.
2. For EACH sentence, search for supporting evidence in the Source. The supporting evidence can be rephrases.
   - IMPORTANT: Look inside headers, lists, and formatted text.
   - IMPORTANT: Accept synonyms (e.g., "capital" matches "capoluogo", "born in" matches dates in brackets).
3. If a claim is not found, verify if it is a logical deduction from the context.
4. If you cannot find a match, explain WHY (e.g. "Masuccio isn't mention anywhere in the source")

Criteria:
- 1: Major Hallucination (Contains specific dates, names, or numbers NOT in the source).
- 3: Mostly faithful, but includes some minor unverifiable details or exaggerations.
- 5: Perfectly Faithful (Every piece of information is supported by the source text, headers, or lists).

Output format:
- Analysis: [Map each summary sentence to a source snippet].
- Verdict: [Explain the score].
- Final: "Score: X" """,

    "completeness": """You are an expert Content Analyst.
Task: Evaluate the Completeness (1-5) of the Summary.
GOAL: Determine if the summary captures the MAIN EVENT/TOPIC of the source, ignoring minor details.

Input Data:
<source_text>
{source}
</source_text>

<summary_text>
{generated}
</summary_text>

Evaluation Steps:
1. Analyze the Source Text and identify the most critical "Who/Where/When/Why".
   - Ignore metadata, file names, or minor side notes.
2. Check if these SPECIFIC core facts are present in the Summary.
   - Allow for rephrasing (e.g., if source says "died in 1990", summary saying "passed away in the 90s" is acceptable coverage).
3. Determine if the summary is completed or leaves the reader confused.

Criteria:
- 1: Irrelevant. Misses the main topic completely.
- 3: Partial. Mentions the topic but misses a crucial fact (e.g., who did it, or the main result).
- 5: Comprehensive. Covers all core entities and main events described in the source.

Output format:
- Key factor identified: [List the Key Facts found in Source].
- Presence in the summary: [YES/NO].
- Final: "Score: X" """,

    "conciseness": """You are an expert Editor.
Task: Evaluate the Conciseness (1-5) of the Summary.
GOAL: Determine if the summary is efficient and dense, or verbose and repetitive.

Input Data:
<summary_text>
{generated}
</summary_text>

Evaluation Steps:
1. Check for repetitive phrases or redundant adjectives.
2. Check if the sentence structure is unnecessarily complex.
3. Verify if the summary packs a lot of information into few words (High Density).

Criteria:
- 1: Verbose/Repetitive. Uses 20 words where 5 would do. Repeats the same information.
- 3: Average. Readable but contains some filler words or slight redundancy.
- 5: Highly Concise. Every word serves a purpose. High information density.

Output format:
- Analysis: [Brief comment on style].
- Final: "Score: X" """,

    "abstraction": """You are an expert Linguist.
Task: Evaluate the Abstraction (1-5) of the Summary.
GOAL: Determine if the model wrote a new text (Abstractive) or just copied sentences (Extractive).

Input Data:
<source_text>
{source}
</source_text>

<summary_text>
{generated}
</summary_text>

Evaluation Steps:
1. Compare the vocabulary and sentence structure of the Source and Summary.
2. Look for "n-gram overlap". Are there long sequences of words identical to the source?
3. Did the model synthesize information (combine two source sentences into one summary sentence)?

Criteria:
- 1: Copy-Paste. The summary copies long phrases (10+ words) entences taken from the source.
- 3: Mixed. Some rephrasing, but relies heavily on original phrasing/cliches.
- 5: Highly Abstractive. The summary uses entirely new vocabulary and sentence structures to convey the same meaning.

Output format:
- Analysis: [Check for copied phrases]
- Final: "Score: X" """
}

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ==========================================
# SEZIONE 3: CONFIGURAZIONE QWEN (vLLM) - OTTIMIZZATA PER TESTI LUNGHI
# ==========================================
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer

# Usiamo Qwen 2.5 14B AWQ
MODEL_ID_Qwen = "Qwen/Qwen2.5-14B-Instruct-AWQ"
MODEL_ID_Llama = "hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4"

print(f"Avvio del motore vLLM con {MODEL_ID_Llama}...")

print(f"Caricamento Tokenizer per {MODEL_ID_Llama}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID_Llama)

# CALCOLO DELLA MEMORIA:
# Qwen 14B AWQ pesa circa 9-10 GB.
# La GPU T4 ha 16 GB.
# Aumentando max_model_len a 4096 usiamo circa 13-14 GB totali.
# Questo permette di leggere testi source molto più lunghi senza troncamenti.

'''llm = LLM(
    model=MODEL_ID_Qwen,
    quantization="awq",
    dtype="half",
    max_model_len=4096,           # <--- AUMENTATO DA 2048 A 4096
    gpu_memory_utilization=0.98,  # <--- SPINGIAMO LA GPU AL LIMITE (98%)
    enforce_eager=True,
    trust_remote_code=True,
    max_num_seqs=16               # Riduciamo leggermente il parallelismo per salvare RAM per il testo
)
'''

llm = LLM(
    model=MODEL_ID_Llama,
    quantization="awq",
    dtype="half",
    max_model_len=16384,           # <--- AUMENTATO A 16K
    gpu_memory_utilization=0.90,
    enforce_eager=True,
    trust_remote_code=True,
    max_num_seqs=32               # Riduciamo leggermente il parallelismo per salvare RAM per il testo
)

terminators = [
    tokenizer.eos_token_id,
    tokenizer.convert_tokens_to_ids("<|eot_id|>")
]
sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=512,               # Aumentato per permettere analisi più dettagliate
    stop_token_ids = terminators
)

print("Motore Llama caricato con contesto esteso (4k)!")

Avvio del motore vLLM con hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4...
Caricamento Tokenizer per hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4...
INFO 02-12 17:41:49 [utils.py:261] non-default args: {'trust_remote_code': True, 'dtype': 'half', 'max_model_len': 16384, 'max_num_seqs': 32, 'disable_log_stats': True, 'quantization': 'awq', 'enforce_eager': True, 'model': 'hugging-quants/Meta-Llama-3.1-8B-Instruct-AWQ-INT4'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 02-12 17:41:51 [model.py:541] Resolved architecture: LlamaForCausalLM
INFO 02-12 17:41:51 [model.py:1561] Using max model len 16384
INFO 02-12 17:41:53 [awq_marlin.py:166] Detected that the model can run with awq_marlin, however you specified quantization=awq explicitly, so forcing awq. Use quantization=awq_marlin for faster inference
INFO 02-12 17:41:53 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.


Parse safetensors files:   0%|          | 0/2 [00:00<?, ?it/s]

INFO 02-12 17:41:54 [vllm.py:624] Asynchronous scheduling is enabled.
WARNING 02-12 17:41:54 [vllm.py:662] Enforce eager set, overriding optimization level to -O0
INFO 02-12 17:41:54 [vllm.py:762] Cudagraph is disabled under eager mode
WARNING 02-12 17:41:56 [system_utils.py:140] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
INFO 02-12 17:42:33 [llm.py:343] Supported tasks: ['generate']
Motore Llama caricato con contesto esteso (4k)!


In [ ]:
# ==========================================
# SEZIONE 4: PIPELINE DI VALUTAZIONE (CORRETTA E COMPLETA)
# ==========================================
import json
import re
import numpy as np

# 0. CONFIGURAZIONE FONDAMENTALE (Ricarichiamo il tokenizer per sicurezza)

# 1. Caricamento Dati
with open(INPUT_PATH, 'r', encoding='utf-8') as f:
    full_data = json.load(f)

data_list = full_data.get("samples", [full_data])
print(f"Caricati {len(data_list)} campioni da valutare.")

# 2. Preparazione del Batch Gigante
prompts_batch = []
mapping_indices = []
metric_order = ["faithfulness", "completeness", "conciseness", "abstraction"]

print("Preparazione dei prompt...")
for i, item in enumerate(data_list):
    source = item.get("source", "")
    reference = item.get("reference", "")
    generated = item.get("generated_summary", "")

    # Inizializziamo/Resettiamo il dizionario 'scores' per sovrascrivere i vecchi dati
    if "scores" not in item:
        item["scores"] = {}

    # Creiamo i 4 prompt
    raw_prompts = [
        GEVAL_PROMPTS["faithfulness"].format(source=source, generated=generated),
        GEVAL_PROMPTS["completeness"].format(source=source, generated=generated),
        GEVAL_PROMPTS["conciseness"].format(generated=generated),
        GEVAL_PROMPTS["abstraction"].format(source=source, generated=generated)
    ]

    for prompt_text in raw_prompts:
        # Ora 'tokenizer' è definito e non darà errore
        formatted_prompt = tokenizer.apply_chat_template(
            [{"role": "system", "content": "You are a strict judge."},
             {"role": "user", "content": prompt_text}],
            tokenize=False,
            add_generation_prompt=True
        )
        prompts_batch.append(formatted_prompt)
        mapping_indices.append(i)

print(f"Totale prompt da elaborare: {len(prompts_batch)}")

# 3. Inferenza Parallela in mini-batch per evitare OOM
print("Inizio generazione vLLM in mini-batch...")
all_outputs = []
BATCH_SIZE = 16 # Adjust this based on your GPU memory and prompt lengths, reduced to 4 from 16

for i in range(0, len(prompts_batch), BATCH_SIZE):
    mini_batch_prompts = prompts_batch[i:i+BATCH_SIZE]
    print(f"Processing mini-batch {i//BATCH_SIZE + 1}/{(len(prompts_batch) + BATCH_SIZE - 1) // BATCH_SIZE} with {len(mini_batch_prompts)} prompts...")
    # Assicurati che 'llm' sia attivo (dalla Sezione 3). Se dà errore qui, riesegui la Sez 3.
    outputs = llm.generate(mini_batch_prompts, sampling_params)
    all_outputs.extend(outputs)

# 4. Parsing e Sovrascrittura Risultati
# ==========================================
# SEZIONE 4, 5, 6: PARSING, OVERALL E SALVATAGGIO
# ==========================================
print("Analisi, sovrascrittura e calcolo Overall...")

# Definiamo i nomi delle metriche
metric_order = ["faithfulness", "completeness", "conciseness", "abstraction"]
judge_keys = [f"judge_{m}" for m in metric_order]

# Inizializza statistiche
all_scores_stats = {key: [] for key in judge_keys}

# --- 1. PARSING DEI VOTI RAW ---
for k, output in enumerate(all_outputs):
    sample_idx = mapping_indices[k]
    base_metric_name = metric_order[k % 4]
    judge_metric_name = judge_keys[k % 4]

    generated_text = output.outputs[0].text.strip()

    # LOGICA DI PARSING BLINDATA (Risolve il problema del count 95/96)
    # Cerca: "Score: 5", "Score: [5]", "**5**", "5."
    matches = re.findall(r'Score[:\s]*\**\[?([1-5])\]?\**', generated_text, re.IGNORECASE)

    if not matches:
        # Fallback: cerca l'ultimo numero isolato (1-5)
        matches = re.findall(r'\b([1-5])\b', generated_text)

    score = int(matches[-1]) if matches else -1

    # Salvataggio nel dizionario del sample
    data_list[sample_idx]["scores"][judge_metric_name] = score
    data_list[sample_idx]["scores"][f"{judge_metric_name}_reason"] = generated_text

    if score != -1:
        all_scores_stats[judge_metric_name].append(score)
    else:
        print(f"⚠️ Warning: Parsing fallito per {judge_metric_name} nel sample {sample_idx}")

# --- 2. CALCOLO 'JUDGE_OVERALL' PER OGNI SAMPLE ---
print("Calcolo delle medie Overall...")
overall_scores_list = []

for item in data_list:
    scores_dict = item.get("scores", {})

    # Raccogliamo i voti validi per questo riassunto
    valid_values = []
    for key in judge_keys:
        val = scores_dict.get(key, -1)
        if val > 0: # Ignora i -1 (errori) e gli 0
            valid_values.append(val)

    # Calcolo Media Aritmetica
    if valid_values:
        avg_score = sum(valid_values) / len(valid_values)
        # Arrotondiamo a 2 decimali per pulizia
        avg_score = round(avg_score, 2)

        item["scores"]["judge_overall"] = avg_score
        overall_scores_list.append(avg_score)
    else:
        item["scores"]["judge_overall"] = -1

# Aggiungiamo la lista dei voti overall alle statistiche globali
all_scores_stats["judge_overall"] = overall_scores_list

# --- 3. AGGIORNAMENTO METRICHE GLOBALI ---
if "metrics" not in full_data: full_data["metrics"] = {}

print("\n--- RISULTATI FINALI (Con Overall) ---")
for m_name, scores in all_scores_stats.items():
    if scores:
        full_data["metrics"][m_name] = {
            "mean": float(np.mean(scores)),
            "std": float(np.std(scores)),
            "count": len(scores)
        }
        print(f" -> {m_name}: Mean {np.mean(scores):.2f}")

# --- 4. SCRITTURA SU FILE ---
full_data["samples"] = data_list
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(full_data, f, ensure_ascii=False, indent=2)

print(f"\nCOMPLETATO! File salvato in: {OUTPUT_PATH}")

Caricati 96 campioni da valutare.
Preparazione dei prompt...
Totale prompt da elaborare: 384
Inizio generazione vLLM in mini-batch...
Processing mini-batch 1/24 with 16 prompts...


Adding requests:   0%|          | 0/16 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/16 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processing mini-batch 2/24 with 16 prompts...


Adding requests:   0%|          | 0/16 [00:00<?, ?it/s]

...

Processed prompts:   0%|          | 0/16 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Processing mini-batch 24/24 with 16 prompts...


Adding requests:   0%|          | 0/16 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/16 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Analisi, sovrascrittura e calcolo Overall...
⚠️ Warning: Parsing fallito per judge_abstraction nel sample 2
⚠️ Warning: Parsing fallito per judge_completeness nel sample 93
Calcolo delle medie Overall...

--- RISULTATI FINALI (Con Overall) ---
 -> judge_faithfulness: Mean 4.01
 -> judge_completeness: Mean 4.33
 -> judge_conciseness: Mean 2.68
 -> judge_abstraction: Mean 2.98
 -> judge_overall: Mean 3.49

COMPLETATO! File salvato in: /content/drive/MyDrive/results_evaluated_llama_vllm.json
